# 作业一：实现HMM中文分词和BPE英文分词
姓名：王凯灵
学号：521030910356

## 任务一：HMM模型用于中文分词

任务一评分标准：
1. 共有8处TODO需要填写，每个TODO计1-2分，共9分，预计代码量30行；
2. 允许自行修改、编写代码完成，对于该情况，请补充注释以便于评分，否则结果不正确将导致较多的扣分；
3. 用于说明实验的文字和总结不额外计分，但不写会导致扣分。

注：本任务仅在短句子上进行效果测试，因此对概率的计算可直接进行连乘。在实践中，常先对概率取对数，将连乘变为加法来计算，以避免出现数值溢出的情况。

导入HMM参数，初始化所需的起始概率矩阵，转移概率矩阵，发射概率矩阵

In [1]:
import pickle
import numpy as np

In [2]:
with open("hmm_parameters.pkl", "rb") as f:
    hmm_parameters = pickle.load(f)

# 非断字（B）为第0行，断字（I）为第1行
# 发射概率矩阵中，词典大小为65536，以汉字的ord作为行key
start_probability = hmm_parameters["start_prob"]  # shape(2,)
trans_matrix = hmm_parameters["trans_mat"]  # shape(2, 2)
emission_matrix = hmm_parameters["emission_mat"]  # shape(2, 65536)

定义待处理的句子

In [3]:
# TODO: 将input_sentence中的xxx替换为你的姓名（1分）
input_sentence = "王凯灵是一名优秀的学生"

实现viterbi算法，并以此进行中文分词

In [31]:
def viterbi(sent_orig, start_prob, trans_mat, emission_mat):
    """
    viterbi算法进行中文分词

    Args:
        sent_orig: str - 输入的句子
        start_prob: numpy.ndarray - 起始概率矩阵
        trans_mat: numpy.ndarray - 转移概率矩阵
        emission_mat: numpy.ndarray - 发射概率矩阵

    Return:
        str - 中文分词的结果
    """

    #  将汉字转为数字表示
    sent_ord = [ord(x) for x in sent_orig]

    # `dp`用来储存不同位置每种标注（B/I）的最大概率值
    dp = np.zeros((2, len(sent_ord)), dtype=float)

    # `path`用来储存最大概率对应的上步B/I选择
    #  例如 path[1][7] == 1 意味着第8个（从1开始计数）字符标注I对应的最大概率，其前一步的隐状态为1（I）
    #  例如 path[0][5] == 1 意味着第6个字符标注B对应的最大概率，其前一步的隐状态为1（I）
    #  例如 path[1][1] == 0 意味着第2个字符标注I对应的最大概率，其前一步的隐状态为0（B）
    path = np.zeros((2, len(sent_ord)), dtype=int)

    #  TODO: 第一个位置的最大概率值计算（1分）
    dp[:, 0] = start_prob * emission_mat[:, sent_ord[0]] # 简化主义

    #  TODO: 其余位置的最大概率值计算（填充dp和path矩阵）（2分）
    for t in range(1, len(sent_ord)):
        max_mat = [[dp[0, t-1] * trans_mat[0, i] * emission_mat[i, sent_ord[t]], dp[1, t-1] * trans_mat[1, i] * emission_mat[i, sent_ord[t]]] for i in range(2)]
        dp[:, t] = np.max(max_mat, axis=1)
        path[:, t] = np.argmax(max_mat, axis=1)

    #  `labels`用来储存每个位置最有可能的隐状态
    labels = [0 for _ in range(len(sent_ord))]

    #  TODO: 计算labels每个位置上的值（填充labels矩阵）（1分）
    labels[-1] = np.argmax(dp[:, -1])
    for t in range(len(sent_ord)-2, -1, -1):
        labels[t] = path[labels[t+1], t+1]

    #  根据lalels生成切分好的字符串
    sent_split = []
    for idx, label in enumerate(labels):
        if label == 1:
            sent_split += [sent_ord[idx], ord("/")]
        else:
            sent_split += [sent_ord[idx]]
    sent_split_str = "".join([chr(x) for x in sent_split])

    return sent_split_str

In [33]:
print("viterbi算法分词结果：", viterbi(input_sentence, start_probability, trans_matrix, emission_matrix))

viterbi算法分词结果： 王凯/灵是/一名/优秀/的/学生/


实现前向算法，计算该句子的概率值

In [6]:
def compute_prob_by_forward(sent_orig, start_prob, trans_mat, emission_mat):
    """
    前向算法，计算输入中文句子的概率值

    Args:
        sent_orig: str - 输入的句子
        start_prob: numpy.ndarray - 起始概率矩阵
        trans_mat: numpy.ndarray - 转移概率矩阵
        emission_mat: numpy.ndarray - 发射概率矩阵

    Return:
        float - 概率值
    """

    #  将汉字转为数字表示
    sent_ord = [ord(x) for x in sent_orig]

    # `dp`用来储存不同位置每种隐状态（B/I）下，到该位置为止的句子的概率
    dp = np.zeros((2, len(sent_ord)), dtype=float)

    # TODO: 初始位置概率的计算（1分）
    dp[:, 0] = start_prob * emission_mat[:, sent_ord[0]]

    # TODO: 先计算其余位置的概率（填充dp矩阵），然后return概率值（1分）
    for t in range(1, len(sent_ord)):
        dp[:, t] = [(dp[0][t-1]*trans_mat[0][j] + dp[1][t-1]*trans_mat[1][j]) * emission_mat[j][sent_ord[t]] for j in range(2)]


    return sum([dp[i][len(sent_ord) - 1] for i in range(2)])

实现后向算法，计算该句子的概率值

In [7]:
def compute_prob_by_backward(sent_orig, start_prob, trans_mat, emission_mat):
    """
    后向算法，计算输入中文句子的概率值

    Args:
        sent_orig: str - 输入的句子
        start_prob: numpy.ndarray - 起始概率矩阵
        trans_mat: numpy.ndarray - 转移概率矩阵
        emission_mat: numpy.ndarray - 发射概率矩阵

    Return:
        float - 概率值
    """

    #  将汉字转为数字表示
    sent_ord = [ord(x) for x in sent_orig]

    # `dp`用来储存不同位置每种隐状态（B/I）下，从结尾到该位置为止的句子的概率
    dp = np.zeros((2, len(sent_ord)), dtype=float)

    # TODO: 终末位置概率的初始化（1分）
    dp[:, -1] = 1

    # TODO: 先计算其余位置的概率（填充dp矩阵），然后return概率值（1分）
    for t in range(len(sent_ord)-2, -1, -1):
        dp[:, t] = [trans_mat[k, 0] * dp[0, t+1] * emission_mat[0, sent_ord[t+1]] + trans_mat[k, 1] * dp[1, t+1] * emission_mat[1, sent_ord[t+1]] for k in range(2)]



    return sum([dp[i][0] * start_prob[i] * emission_mat[i][sent_ord[0]] for i in range(2)])

In [8]:
print("前向算法概率：", compute_prob_by_forward(input_sentence, start_probability, trans_matrix, emission_matrix))
print("后向算法概率：", compute_prob_by_backward(input_sentence, start_probability, trans_matrix, emission_matrix))

前向算法概率： 2.496754973952008e-34
后向算法概率： 2.4967549739520076e-34


（TODO：实验总结）

Viterbi和前后向算法本质上都是动态规划方法，通过更新dp矩阵获取最大概率的句子，只是更新规则不同。主要区别在于Viterbi算法是为了找到最有可能的隐藏状态序列，而前后向算法是为了计算给定观察序列的概率，并得到某个时间点上的状态分布。

TODO2-4实现Viterbi算法，按照习惯尽可能使用矩阵运算减少代码量与提高效率。就结果而言有时不能正确的分开姓名，但更改名字，如改为小明等简单名字总是可以正确分出“小明/是/一名/优秀/的/学生/”，有些名字配合“不是”也能被正确分出。实现时有一个需要注意的细节：是否需要乘以emission_mat以后再取最大值。我不太记得上课讲的版本了，wiki采用的是先乘以后取最大。

TODO5-8实现前后向算法，只实现了前向与后向的概率计算。按照算法步骤实现即可，可以观察到前后向概率相同的性质。

## 任务二：BPE算法用于英文分词

任务二评分标准：

1. 共有7处TODO需要填写，每个TODO计1-2分，共9分，预计代码量50行；
2. 允许自行修改、编写代码完成，对于该情况，请补充注释以便于评分，否则结果不正确将导致较多的扣分；
3. 用于说明实验的文字和总结不额外计分，但不写会导致扣分。

构建空格分词器，将语料中的句子以空格切分成单词，然后将单词拆分成字母加`</w>`的形式。例如`apple`将变为`a p p l e </w>`。

In [9]:
import re
import functools

In [10]:
_splitor_pattern = re.compile(r"[^a-zA-Z']+|(?=')")
_digit_pattern = re.compile(r"\d+")


def white_space_tokenize(corpus):
    """
    先正则化（字母转小写、数字转为N、除去标点符号），然后以空格分词语料中的句子，例如：
    输入 corpus=["I am happy.", "I have 10 apples!"]，
    得到 [["i", "am", "happy"], ["i", "have", "N", "apples"]]

    Args:
        corpus: List[str] - 待处理的语料

    Return:
        List[List[str]] - 二维List，内部的List由每个句子的单词str构成
    """

    tokeneds = [list(filter(lambda tkn: len(tkn) > 0, _splitor_pattern.split(_digit_pattern.sub("N", stc.lower())))) for stc in corpus]

    return tokeneds

编写相应函数构建BPE算法需要用到的初始状态词典

In [11]:
def build_bpe_vocab(corpus):
    """
    将语料进行white_space_tokenize处理后，将单词每个字母以空格隔开、结尾加上</w>后，构建带频数的字典，例如：
    输入 corpus=["I am happy.", "I have 10 apples!"]，
    得到
    {
        'i </w>': 2,
        'a m </w>': 1,
        'h a p p y </w>': 1,
        'h a v e </w>': 1,
        'N </w>': 1,
        'a p p l e s </w>': 1
     }

    Args:
        corpus: List[str] - 待处理的语料

    Return:
        Dict[str, int] - "单词分词状态->频数"的词典
    """

    tokenized_corpus = white_space_tokenize(corpus)

    bpe_vocab = dict()

    # TODO: 完成函数体（1分）
    for s in tokenized_corpus:
        for w in s:
            key = ' '.join(w) + ' </w>'
            bpe_vocab[key] = bpe_vocab.get(key, 0) + 1

    return bpe_vocab

In [12]:
def test_bpe_vocab():
    corpus = ["I am happy.", "I have 10 apples!"]
    test_bpe_vocab = build_bpe_vocab(corpus)
    for elem in test_bpe_vocab.items():
        print(elem)
        
test_bpe_vocab()

('i </w>', 2)
('a m </w>', 1)
('h a p p y </w>', 1)
('h a v e </w>', 1)
('N </w>', 1)
('a p p l e s </w>', 1)


编写所需的其他函数

In [13]:
def get_bigram_freq(bpe_vocab):
    """
    统计"单词分词状态->频数"的词典中，各bigram的频次（假设该词典中，各个unigram以空格间隔），例如：
    输入 bpe_vocab=
    {
        'i </w>': 2,
        'a m </w>': 1,
        'h a p p y </w>': 1,
        'h a v e </w>': 1,
        'N </w>': 1,
        'a p p l e s </w>': 1
    }
    得到
    {
        ('i', '</w>'): 2,
        ('a', 'm'): 1,
        ('m', '</w>'): 1,
        ('h', 'a'): 2,
        ('a', 'p'): 2,
        ('p', 'p'): 2,
        ('p', 'y'): 1,
        ('y', '</w>'): 1,
        ('a', 'v'): 1,
        ('v', 'e'): 1,
        ('e', '</w>'): 1,
        ('N', '</w>'): 1,
        ('p', 'l'): 1,
        ('l', 'e'): 1,
        ('e', 's'): 1,
        ('s', '</w>'): 1
    }

    Args:
        bpe_vocab: Dict[str, int] - "单词分词状态->频数"的词典

    Return:
        Dict[Tuple(str, str), int] - "bigram->频数"的词典
    """

    bigram_freq = dict()

    # TODO: 完成函数体（1分）
    for word, freq in bpe_vocab.items():
        word_split = word.split(' ')
        for i in range(len(word_split) - 1):
            bigram = (word_split[i], word_split[i+1])
            bigram_freq[bigram] = bigram_freq.get(bigram, 0) + freq

    return bigram_freq

In [14]:
def test_get_bigram_freq():
    corpus = ["I am happy.", "I have 10 apples!"]
    test_bigram_freq = get_bigram_freq(build_bpe_vocab(corpus))
    for elem in test_bigram_freq.items():
        print(elem)
        
test_get_bigram_freq()

(('i', '</w>'), 2)
(('a', 'm'), 1)
(('m', '</w>'), 1)
(('h', 'a'), 2)
(('a', 'p'), 2)
(('p', 'p'), 2)
(('p', 'y'), 1)
(('y', '</w>'), 1)
(('a', 'v'), 1)
(('v', 'e'), 1)
(('e', '</w>'), 1)
(('N', '</w>'), 1)
(('p', 'l'), 1)
(('l', 'e'), 1)
(('e', 's'), 1)
(('s', '</w>'), 1)


In [50]:
def refresh_bpe_vocab_by_merging_bigram(bigram, old_bpe_vocab):
    """
    在"单词分词状态->频数"的词典中，合并指定的bigram（即去掉对应的相邻unigram之间的空格），最后返回新的词典，例如：
    输入 bigram=('i', '</w>')，old_bpe_vocab=
    {
        'i </w>': 2,
        'a m </w>': 1,
        'h a p p y </w>': 1,
        'h a v e </w>': 1,
        'N </w>': 1,
        'a p p l e s </w>': 1
    }
    得到
    {
        'i</w>': 2,
        'a m </w>': 1,
        'h a p p y </w>': 1,
        'h a v e </w>': 1,
        'N </w>': 1,
        'a p p l e s </w>': 1
    }

    Args:
        old_bpe_vocab: Dict[str, int] - 初始"单词分词状态->频数"的词典

    Return:
        Dict[str, int] - 合并后的"单词分词状态->频数"的词典
    """

    new_bpe_vocab = dict()

    # TODO: 完成函数体（1分）
    # bigram_str = ' '.join(bigram)
    merged_str = ''.join(bigram)

    # for word, freq in old_bpe_vocab.items():
    #     new_word = word.replace(bigram_str, merged_str)
    #     new_bpe_vocab[new_word] = freq

    for word, freq in old_bpe_vocab.items():
        word_split = word.split(' ')
        new_word = []
        i = 0
        while i < len(word_split):
            if i < len(word_split) - 1 and tuple(word_split[i:i+2]) == bigram:
                new_word.append(merged_str)
                i += 2
            else:
                new_word.append(word_split[i])
                i += 1
        new_bpe_vocab[' '.join(new_word)] = freq
    
    return new_bpe_vocab

In [51]:
refresh_bpe_vocab_by_merging_bigram(bigram=('h', 'a'), 
    old_bpe_vocab= {
            'i </w>': 2,
            'a m </w>': 1,
            'h a p p y </w>': 1,
            'h a v e </w>': 1,
            'N </w>': 1,
            'a p p l e s </w>': 1
            })

{'i </w>': 2,
 'a m </w>': 1,
 'ha p p y </w>': 1,
 'ha v e </w>': 1,
 'N </w>': 1,
 'a p p l e s </w>': 1}

In [52]:
def get_bpe_tokens(bpe_vocab):
    """
    根据"单词分词状态->频数"的词典，返回所得到的BPE分词列表，并将该列表按照分词长度降序排序返回，例如：
    输入 bpe_vocab=
    {
        'i</w>': 2,
        'a m </w>': 1,
        'ha pp y </w>': 1,
        'ha v e </w>': 1,
        'N </w>': 1,
        'a pp l e s </w>': 1
    }
    得到
    [
        ('i</w>', 2),
        ('ha', 2),
        ('pp', 2),
        ('a', 2),
        ('m', 1),
        ('</w>', 5),
        ('y', 1),
        ('v', 1),
        ('e', 2),
        ('N', 1),
        ('l', 1),
        ('s', 1)
     ]

    Args:
        bpe_vocab: Dict[str, int] - "单词分词状态->频数"的词典

    Return:
        List[Tuple(str, int)] - BPE分词和对应频数组成的List
    """

    # TODO: 完成函数体（2分）
    token_freq = dict()

    for word, freq in bpe_vocab.items():
        tokens = word.split()
        for token in tokens:
            token_freq[token] = token_freq.get(token, 0) + freq

    bpe_tokens = sorted(token_freq.items(), key=lambda x: (-len(x[0]) if x[0][-4:]!='</w>' else 3-len(x[0]), x[0]))


    return bpe_tokens

In [53]:
get_bpe_tokens(bpe_vocab=
    {
        'i</w>': 2,
        'a m </w>': 1,
        'ha pp y </w>': 1,
        'ha v e </w>': 1,
        'N </w>': 1,
        'a pp l e s </w>': 1
    })

[('ha', 2),
 ('i</w>', 2),
 ('pp', 2),
 ('</w>', 5),
 ('N', 1),
 ('a', 2),
 ('e', 2),
 ('l', 1),
 ('m', 1),
 ('s', 1),
 ('v', 1),
 ('y', 1)]

In [54]:
def print_bpe_tokenize(word, bpe_tokens):
    """
    根据按长度降序的BPE分词列表，将所给单词进行BPE分词，最后打印结果。
    
    思想是，对于一个待BPE分词的单词，按照长度顺序从列表中寻找BPE分词进行子串匹配，
    若成功匹配，则对该子串左右的剩余部分递归地进行下一轮匹配，直到剩余部分长度为0，
    或者剩余部分无法匹配（该部分整体由"<unknown>"代替）
    
    例1：
    输入 word="supermarket", bpe_tokens=[
        ("su", 20),
        ("are", 10),
        ("per", 30),
    ]
    最终打印 "su per <unknown>"

    例2：
    输入 word="shanghai", bpe_tokens=[
        ("hai", 1),
        ("sh", 1),
        ("an", 1),
        ("</w>", 1),
        ("g", 1)
    ]
    最终打印 "sh an g hai </w>"

    Args:
        word: str - 待分词的单词str
        bpe_tokens: List[Tuple(str, int)] - BPE分词和对应频数组成的List
    """

    # TODO: 请尝试使用递归函数定义该分词过程（2分）
    sorted_bpe_tokens = sorted(bpe_tokens, key=lambda x: len(x[0]), reverse=True)
    def bpe_tokenize(sub_word):
        for token, _ in sorted_bpe_tokens:
            if token in sub_word:
                parts = sub_word.split(token, 1)
                return bpe_tokenize(parts[0]) + token + ' ' + bpe_tokenize(parts[1])
        
        return '<unknown> ' if sub_word else ''

    res = bpe_tokenize(word + "</w>")
    print(res)

开始读取数据集并训练BPE分词器

In [55]:
print_bpe_tokenize("supermarket", [("su", 20), ("are", 10), ("per", 30)])
print_bpe_tokenize("shanghai", [("hai", 1), ("sh", 1), ("an", 1), ("</w>", 1), ("g", 1)])
print_bpe_tokenize("shanghaishang", [("hai", 1), ("sh", 1), ("an", 1), ("</w>", 1), ("g", 1)])

su per <unknown> 
sh an g hai </w> 
sh an g hai sh an g </w> 


In [56]:
with open("data/news.2007.en.shuffled.deduped.train", encoding="utf-8") as f:
    training_corpus = list(map(lambda l: l.strip(), f.readlines()[:1000]))

print("Loaded training corpus.")

Loaded training corpus.


In [57]:
training_iter_num = 300

training_bpe_vocab = build_bpe_vocab(training_corpus)
for i in range(training_iter_num):
    # TODO: 完成训练循环内的代码逻辑（2分）
    bigram_freq = get_bigram_freq(training_bpe_vocab)
    most_freq_bigram = max(bigram_freq, key=bigram_freq.get)
    training_bpe_vocab = refresh_bpe_vocab_by_merging_bigram(most_freq_bigram, training_bpe_vocab)

training_bpe_tokens = get_bpe_tokens(training_bpe_vocab)

测试BPE分词器的分词效果

In [59]:
test_word = "naturallanguageprocessing"

print("naturallanguageprocessing 的分词结果为：")
print_bpe_tokenize(test_word, training_bpe_tokens)

naturallanguageprocessing 的分词结果为：
n a tur all an g u ag e pro c es s ing</w> 


（TODO：实验总结）

TODO1：只是简单的字符串处理。
定义了test_bpe_vocab函数使用给出例子进行测试。结果正确。

TODO2: 亦为简单字符串处理，在test_get_bigram_freq函数中测试正确。

TODO3：原本想使用内置函数replane简单实现，但结果与所给示例不同，故手动实现。仍然进行正确性测试，但此例给的输入结果并不改变，令人疑惑。

TODO4：测试结果正常，但顺序与例子不同，暂时还为推测出例子所用的实现方法。但不影响所需功能，故不做处理。

TODO5：要求使用递归写。首先按要求对长度排序。上课时强调鼓励使用工具进行辅助，于是与GPT交流。在未进行提示时，GPT给出的方法是匹配分割后从分割点左右递归，虽然在本例子中不产生影响，但实际上匹配次数可以大于1，于是进行纠正。纠正后GPT希望使用list保存所有位置，在所有位置两侧进行递归。这是错误的，正确方法应该是对每个匹配间的子段递归。随即想到既然每次递归都遍历token，其实只需要在第一次匹配进行左右递归即可。此外，在我要求进行代码优化时，GPT提出使用外部字典cache split的结果。看似花里胡哨但对于长段落十分合理，工程上的意义有待确认。

TODO6：用之前的函数获取并根据频率排序bigram，找出最高频的bigram并更新training_bpe_vocab即可。分词效果令人震撼。

至此，一共填写了6处TODO，但要求为7处，不过确实是9分。令人疑惑。